# 04 Win Rate — Logistic Regression

## Goal

For every `(character, card)` pair in `gold_card_choice_events`, restrict to occasions where the card was offered and fit

```
victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level
```

`was_picked`'s coefficient is the card's effect on win probability *holding run state at the time of the offer constant* — a cleaner signal than the raw pick/no-pick lift computed in 03, which doesn't control for anything.

This fits every card directly rather than pre-selecting a curated "cards of interest" list by raw lift and then re-estimating on the same data. That selection step is a winner's-curse setup: any card's raw lift is true effect plus sampling noise, and sorting by it preferentially keeps cards that got a lucky noise draw — that noise doesn't average out on a second look at the same rows, so the regression would end up biased toward whatever the screen happened to reward. Fitting every card sidesteps the problem entirely instead of working around it with a data split. `03`'s raw-lift screen is still worth checking against the results here as a sanity check (a card with strong raw lift but a controlled odds ratio near 1 suggests the raw lift was confounded), just not as a required input.

**Deliberately excluded:** `floor_reached` and `floors_gained` are not covariates here, even though they're in the table. Both are facts about how the run *ended*, not facts known at the time of the pick — `floor_reached` is close to deterministic of `victory` (a run that reaches floor 57 essentially won), so including it would leak the outcome into the predictors rather than control for a legitimate confounder. `floor` (the pick's own floor, i.e. `choice_floor`) is fine to include — that's "how far into the run this decision happened," known at decision time.

In [1]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

GOLD_CARD_CHOICE_EVENTS_PATH = str(PROJECT_ROOT / "raw_data" / "gold" / "card_choice_events")

In [2]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder.master("local[4]")
    .appName("win-rate-logistic-regression")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "16g")
    .config("spark.driver.maxResultSize", "6g")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(GOLD_CARD_CHOICE_EVENTS_PATH)
print(f"Loaded {GOLD_CARD_CHOICE_EVENTS_PATH}")

Loaded e:\Projects\sts-card-choice-analysis\raw_data\gold\card_choice_events


In [3]:
# Collect per character rather than the whole table in one toPandas() — a single collect
# over every card blew past both spark.driver.maxResultSize and, once that was raised, the
# JVM driver heap itself (collect() stages all rows as boxed Java objects before handing
# them to Python, which is much heavier than the on-disk Parquet size). Card pools are
# already partitioned by character_chosen, so chunking on it is free and bounds peak driver
# memory to roughly one character's worth of rows at a time.
characters = sorted(
    row["character_chosen"]
    for row in df.select("character_chosen").distinct().collect()
    if row["character_chosen"] is not None
)
print("Characters:", characters)

regression_pd_chunks = []
for character in characters:
    chunk = (
        df.filter(F.col("character_chosen") == character)
        .select("character_chosen", "card_name", "was_picked", "victory", "floor", "current_hp", "max_hp", "relic_count", "ascension_level")
        .na.drop()
        .toPandas()
    )
    print(f"{character}: collected {len(chunk)} rows")
    regression_pd_chunks.append(chunk)

regression_pd = pd.concat(regression_pd_chunks, ignore_index=True)
del regression_pd_chunks
print("Collected rows:", len(regression_pd))
print("Distinct (character, card) pairs:", regression_pd[["character_chosen", "card_name"]].drop_duplicates().shape[0])

Characters: ['DEFECT', 'IRONCLAD', 'THE_SILENT', 'WATCHER']
DEFECT: collected 54242228 rows
IRONCLAD: collected 78113867 rows
THE_SILENT: collected 60027797 rows
WATCHER: collected 38082393 rows
Collected rows: 230466285
Distinct (character, card) pairs: 3139


## Stop Spark

Nothing below this point touches `df`/`spark` again — the rest of the notebook works off the
`regression_pd` parquet cache. Stopping here (rather than after the fit, as this used to)
releases the JVM driver's memory before the memory-heavy `groupby().apply()` fit below runs;
leaving Spark's 16g driver resident at the same time as a ~20GB pandas frame was enough to push
the machine into disk-swap territory.

In [ ]:
spark.stop()

In [1]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import statsmodels.formula.api as smf

# pd.read_parquet builds a full Arrow Table, then converts it to pandas - both copies exist in
# memory at once during that conversion, roughly doubling peak memory right at the read step,
# before any downcasting below gets a chance to shrink anything. self_destruct=True frees each
# column's Arrow buffer as soon as it's converted, instead of holding both copies simultaneously.
_table = pq.read_table("../raw_data/win_rate_regression_input.parquet")
regression_pd = _table.to_pandas(self_destruct=True)
del _table
regression_pd["was_picked"] = regression_pd["was_picked"].astype(int)
regression_pd["victory"] = regression_pd["victory"].astype(int)
regression_pd = regression_pd[regression_pd["max_hp"] > 0]
regression_pd["hp_ratio"] = regression_pd["current_hp"] / regression_pd["max_hp"]
regression_pd = regression_pd.drop(columns=["current_hp", "max_hp"])

# Default dtypes (object strings, int64 everywhere) put this 230M-row frame at ~20GB in memory
# - close enough to the RAM ceiling that the groupby/apply fit below can start swapping to disk
# instead of actually computing. Downcast to the smallest dtype that fits each column's range.
regression_pd["character_chosen"] = regression_pd["character_chosen"].astype("category")
regression_pd["card_name"] = regression_pd["card_name"].astype("category")
regression_pd["was_picked"] = regression_pd["was_picked"].astype("int8")
regression_pd["victory"] = regression_pd["victory"].astype("int8")
regression_pd["floor"] = regression_pd["floor"].astype("int16")
regression_pd["relic_count"] = regression_pd["relic_count"].astype("int8")
regression_pd["ascension_level"] = regression_pd["ascension_level"].astype("int8")
regression_pd["hp_ratio"] = regression_pd["hp_ratio"].astype("float32")

MIN_REGRESSION_ROWS = 200

# groupby().apply() needs every group to return a Series with the same keys — a group
# returning fewer keys than another (e.g. just n/error on failure) makes pandas fall back
# to a stacked long-format result instead of one row per group, so every branch below
# fills the full set of keys even when most are None.
EMPTY_RESULT = {
    "n": None,
    "error": None,
    "converged": None,
    "was_picked_coef": None,
    "was_picked_pvalue": None,
    "odds_ratio": None,
    "odds_ratio_ci_low": None,
    "odds_ratio_ci_high": None,
    "pseudo_r2": None,
    "auc": None,
}

def fit_card_logit(group):
    result = dict(EMPTY_RESULT)
    if len(group) < MIN_REGRESSION_ROWS:
        result["n"] = len(group)
        result["error"] = "too few rows"
        return pd.Series(result)
    try:
        model = smf.logit(
            "victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level",
            data=group,
        ).fit(disp=0)
    except Exception as exc:
        result["n"] = len(group)
        result["error"] = str(exc)
        return pd.Series(result)
    coef = model.params["was_picked"]
    ci_low, ci_high = model.conf_int().loc["was_picked"]
    # Small, lopsided groups (a card picked in nearly every winning run and almost never in
    # a losing one) can hit quasi/complete separation: the optimizer keeps pushing
    # was_picked's coefficient toward infinity without the likelihood actually converging.
    # statsmodels doesn't raise for this - .fit() just returns whatever the optimizer had on
    # its last iteration, with a wide, unstable confidence interval. mle_retvals["converged"]
    # is the authoritative flag for this, independent of whether .fit() raised.
    converged = bool(model.mle_retvals.get("converged", False))
    # pseudo_r2 (McFadden's) measures how much of victory's variance the whole model
    # explains; auc measures how well it discriminates wins from losses. Both matter
    # alongside was_picked_pvalue because with n in the hundreds of thousands, trivial
    # effects reach significance — these catch "significant but practically meaningless."
    # auc is scored in-sample (same rows the model was fit on), so it reads as fit quality,
    # not held-out predictive performance - fine for this notebook's inferential use (reading
    # was_picked's coefficient), not a claim about the model's predictive accuracy.
    result.update({
        "n": len(group),
        "error": None,
        "converged": converged,
        "was_picked_coef": coef,
        "was_picked_pvalue": model.pvalues["was_picked"],
        "odds_ratio": np.exp(coef),
        "odds_ratio_ci_low": np.exp(ci_low),
        "odds_ratio_ci_high": np.exp(ci_high),
        "pseudo_r2": model.prsquared,
        "auc": roc_auc_score(group["victory"], model.predict()),
    })
    return pd.Series(result)

results_pd = (
    regression_pd.groupby(["character_chosen", "card_name"], observed=True)
    .apply(fit_card_logit, include_groups=False)
    .reset_index()
)
results_pd["converged"] = results_pd["converged"].astype("boolean")
n_fitted = results_pd["error"].isna().sum()
n_converged = (results_pd["error"].isna() & results_pd["converged"]).sum()
print("Fitted:", n_fitted, "of", len(results_pd))
print("Converged:", n_converged, "of", n_fitted, "fitted")

e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
e:\Projects\sts-card-choice-analysis\.venv\Lib\site-packages\stats

Fitted: 2231 of 3139
Converged: 2221 of 2231 fitted


In [2]:
print("Error breakdown (top 20):")
print(results_pd["error"].value_counts(dropna=False).head(20))
print()
print("Convergence among fitted models:")
print(results_pd.loc[results_pd["error"].isna(), "converged"].value_counts(dropna=False))

Error breakdown (top 20):
error
NaN                2231
too few rows        896
Singular matrix      12
Name: count, dtype: int64

Convergence among fitted models:
converged
True     2221
False      10
Name: count, dtype: Int64


In [3]:
from statsmodels.stats.multitest import multipletests

# 2,232 independent regressions filtered at raw p < 0.05 would let ~5% of cards with no true
# effect (100+ cards) cross significance by chance alone. Benjamini-Hochberg controls the
# expected proportion of false discoveries among the models actually being decided among -
# i.e. the fitted, converged ones; non-converged fits are excluded here too since they're
# dropped from `significant` below regardless of p-value.
testable = results_pd["error"].isna() & results_pd["converged"].fillna(False)
results_pd["was_picked_qvalue"] = np.nan
results_pd.loc[testable, "was_picked_qvalue"] = multipletests(
    results_pd.loc[testable, "was_picked_pvalue"], method="fdr_bh"
)[1]
print(f"{testable.sum()} models entered the FDR correction")

2221 models entered the FDR correction


### Results

`odds_ratio` > 1 means picking the card is associated with higher win odds after controlling for HP ratio, floor, relic count, and ascension at the time it was offered; < 1 means lower.

The `significant` table below is restricted three ways, each guarding against a different way "significant" can be misleading at this scale:

- **`converged`** — drops models where the MLE didn't converge. Small, lopsided card groups (a card picked in nearly every winning run and almost never in a losing one) can hit quasi/complete separation, where the optimizer keeps pushing `was_picked`'s coefficient toward infinity without the likelihood actually converging. statsmodels doesn't raise for this — it returns whatever the optimizer had on its last iteration, with a wide, unstable confidence interval (an odds ratio of 22 with a 95% CI of roughly [1, 400] is the signature of this, not a real effect). `model.mle_retvals["converged"]` is the flag that catches it.
- **`was_picked_qvalue < 0.05`** (Benjamini-Hochberg FDR-adjusted p-value) instead of the raw `was_picked_pvalue` — with 2,232 independent regressions, filtering on the raw p-value at 0.05 would let roughly 5% of cards with no true effect (100+ cards) cross significance by chance alone. BH correction controls the expected false-discovery proportion among the reported results instead.
- **effect size outside `[0.9, 1.1]`** — clearing the q-value bar alone doesn't rule out a statistically real but practically trivial effect (an odds ratio of 1.02 estimated from hundreds of thousands of rows is technically significant and not worth acting on). Requiring the odds ratio to actually move the needle keeps the table focused on effects worth caring about, not just ones large samples can detect.

Cards that don't clear all three bars are dropped from this view but still available in `results_pd`, along with `pseudo_r2` and `auc` for judging each surviving model's overall fit (`auc` is computed in-sample, so it reads as fit quality, not held-out predictive performance).

In [4]:
significant = results_pd[
    results_pd["error"].isna()
    & results_pd["converged"].fillna(False)
    & (results_pd["was_picked_qvalue"] < 0.05)
    & ((results_pd["odds_ratio"] <= 0.9) | (results_pd["odds_ratio"] >= 1.1))
].sort_values(["character_chosen", "odds_ratio"], ascending=[True, False])

cols = ["character_chosen", "card_name", "n", "odds_ratio", "odds_ratio_ci_low", "odds_ratio_ci_high", "was_picked_pvalue", "was_picked_qvalue", "pseudo_r2", "auc"]
significant[cols]

,character_chosen,card_name,n,odds_ratio,odds_ratio_ci_low,odds_ratio_ci_high,was_picked_pvalue,was_picked_qvalue,pseudo_r2,auc
142,DEFECT,Concentrate,313,3.876976,1.566486,9.595329,3.382009e-03,0.010192,0.291626,0.852159
95,DEFECT,Burning Pact,378,3.460880,1.667772,7.181854,8.585789e-04,0.002872,0.260712,0.835572
443,DEFECT,Night Terror,343,2.357926,1.272884,4.367888,6.390193e-03,0.018176,0.258858,0.838504
406,DEFECT,Master of Strategy+1,1206,1.947548,1.506740,2.517317,3.562503e-07,0.000002,0.125937,0.727818
320,DEFECT,Glacier,431375,1.604859,1.576619,1.633605,0.000000e+00,0.000000,-inf,0.758425
...,...,...,...,...,...,...,...,...,...,...
2489,WATCHER,Chaos,355,0.307958,0.116435,0.814515,1.762563e-02,0.043985,0.255643,0.821868
3074,WATCHER,Underhanded Strike+1,602,0.288705,0.107192,0.777584,1.398663e-02,0.036037,0.201367,0.790276
2918,WATCHER,Reinforced Body,313,0.283066,0.124948,0.641278,2.488042e-03,0.007707,0.270698,0.830518
2693,WATCHER,Genetic Algorithm,350,0.263269,0.102977,0.673069,5.325973e-03,0.015382,0.240642,0.817560


In [5]:
summary = (
    significant.groupby("character_chosen")
    .agg(n_significant=("card_name", "count"),
         n_positive=("odds_ratio", lambda s: (s > 1).sum()),
         n_negative=("odds_ratio", lambda s: (s < 1).sum()))
)
summary.to_string()

'                  n_significant  n_positive  n_negative\ncharacter_chosen                                       \nDEFECT                      184          46         138\nIRONCLAD                    230          42         188\nTHE_SILENT                  216          50         166\nWATCHER                     194          34         160'

In [6]:
top = (
    significant.groupby("character_chosen")
    .apply(lambda g: g.nlargest(5, "odds_ratio")[["card_name","n","odds_ratio","was_picked_pvalue"]], include_groups=False)
)
bottom = (
    significant.groupby("character_chosen")
    .apply(lambda g: g.nsmallest(5, "odds_ratio")[["card_name","n","odds_ratio","was_picked_pvalue"]], include_groups=False)
)
print("TOP 5 per character (best odds_ratio):")
print(top.to_string())
print()
print("BOTTOM 5 per character (worst odds_ratio):")
print(bottom.to_string())

TOP 5 per character (best odds_ratio):
                                  card_name       n  odds_ratio  was_picked_pvalue
character_chosen                                                                  
DEFECT           142            Concentrate     313    3.876976       3.382009e-03
                 95            Burning Pact     378    3.460880       8.585789e-04
                 443           Night Terror     343    2.357926       6.390193e-03
                 406   Master of Strategy+1    1206    1.947548       3.562503e-07
                 320                Glacier  431375    1.604859       0.000000e+00
IRONCLAD         1175  Master of Strategy+1    1749    1.860990       1.301075e-08
                 1234            Offering+1   31711    1.849281      1.663752e-134
                 1013             Expertise     689    1.835461       1.970774e-02
                 1174    Master of Strategy   21161    1.730205       7.364947e-38
                 1233              Offering  491

In [7]:
print(regression_pd.memory_usage(deep=True).sum() / 1e9, "GB in memory")
print(len(regression_pd), "rows")

2.996086615 GB in memory
230466285 rows


In [8]:
regression_pd.to_parquet("../raw_data/win_rate_regression_input.parquet", index=False)
results_pd.to_parquet("../raw_data/card_win_rate_regression_results.parquet", index=False)

In [9]:
significant[cols].to_csv("../raw_data/card_win_rate_significant.csv", index=False)
summary.to_csv("../raw_data/card_win_rate_summary.csv")